# 📦 Notebook 1 — Data Preprocessing & ALS Model Training

**Domain:** E-commerce Products (Amazon Reviews)  
**Focus:** Personalization — Cold-Start Problem & User Segmentation  
**Dataset:** Amazon Product Reviews (Electronics) — ~1.6M reviews  

## Architecture Overview
```
Amazon CSV  →  Spark Preprocessing  →  ALS Training  →  Saved Model
                                                              ↓
                                              Notebook 3 (Streaming) loads it
```

## Step 1 — Download the Dataset

We use the **Amazon Product Reviews — Electronics** subset from the UCSD Julian McAuley lab.
It contains ~1.69 million reviews with `(user_id, item_id, rating, timestamp)` — perfect for ALS collaborative filtering.

Why this dataset fits our domain:
- Real e-commerce interaction data
- Sufficient sparsity to challenge collaborative filtering (cold-start is real here)
- Large enough (>500K) to justify distributed Spark processing
- Natural user segments exist (casual vs power shoppers)

In [ ]:
import os, subprocess

DATA_DIR = '/data'
RAW_FILE = f'{DATA_DIR}/ratings_Electronics.csv'

os.makedirs(DATA_DIR, exist_ok=True)

if not os.path.exists(RAW_FILE):
    print('Downloading Amazon Electronics dataset...')
    # Download the gzipped ratings-only file (no review text, just interactions)
    url = 'https://snap.stanford.edu/data/amazon/productGraph/categoryFiles/ratings_Electronics.csv'
    subprocess.run(['wget', '-q', '-O', RAW_FILE, url], check=True)
    print('Download complete.')
else:
    print('Dataset already exists.')

# Count lines to verify size
result = subprocess.run(['wc', '-l', RAW_FILE], capture_output=True, text=True)
print(f'Total rows: {result.stdout.strip()}')

## Step 2 — Start Spark Session

We create a SparkSession connecting to the Spark Master running in the same Docker network.
The `spark://spark-master:7077` URL tells PySpark to use the cluster, not local mode.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName('AmazonRecommender-ALS-Training')
    .master('spark://spark-master:7077')      # Use the Spark cluster
    .config('spark.executor.memory', '1g')
    .config('spark.driver.memory', '1g')
    .config('spark.sql.shuffle.partitions', '8')  # Match worker count × cores
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')  # Reduce noise in output
print(f'Spark version: {spark.version}')
print(f'Spark UI: http://localhost:4040')

## Step 3 — Load & Inspect Raw Data

The Amazon Electronics CSV has 4 columns with NO header:
`user_id (string), item_id (string), rating (float), timestamp (unix int)`

In [ ]:
# Define schema explicitly — faster than inferSchema=True on large files
schema = StructType([
    StructField('user_id_raw',  StringType(),  True),
    StructField('item_id_raw',  StringType(),  True),
    StructField('rating',       FloatType(),   True),
    StructField('timestamp',    LongType(),    True),
])

raw_df = (
    spark.read
    .option('header', 'false')   # No header in this file
    .schema(schema)
    .csv(RAW_FILE)
)

print(f'Total records loaded: {raw_df.count():,}')
raw_df.show(5, truncate=40)

## Step 4 — Data Preprocessing

ALS requires **integer** user and item IDs (not strings). We use `StringIndexer` to map
Amazon's alphanumeric IDs → dense integers. We also:
- Drop nulls and invalid ratings (must be 1.0–5.0)
- Remove duplicate interactions (keep latest)
- Filter users/items with very few interactions (cold-start mitigation for training)

In [ ]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

# ── 4a. Drop nulls ────────────────────────────────────────────────────────
clean_df = raw_df.dropna(subset=['user_id_raw', 'item_id_raw', 'rating', 'timestamp'])
print(f'After dropna: {clean_df.count():,}')

# ── 4b. Validate rating range (1.0 – 5.0) ────────────────────────────────
clean_df = clean_df.filter((F.col('rating') >= 1.0) & (F.col('rating') <= 5.0))
print(f'After rating validation: {clean_df.count():,}')

# ── 4c. Remove duplicate (user, item) pairs — keep the latest interaction ─
window = Window.partitionBy('user_id_raw', 'item_id_raw').orderBy(F.col('timestamp').desc())
clean_df = (
    clean_df
    .withColumn('row_num', F.row_number().over(window))
    .filter(F.col('row_num') == 1)
    .drop('row_num')
)
print(f'After deduplication: {clean_df.count():,}')

# ── 4d. Filter sparse users/items (at least 5 interactions each) ──────────
# This improves ALS quality AND is the first cold-start mitigation step
user_counts = clean_df.groupBy('user_id_raw').count().filter(F.col('count') >= 5)
item_counts = clean_df.groupBy('item_id_raw').count().filter(F.col('count') >= 5)

clean_df = (
    clean_df
    .join(user_counts.select('user_id_raw'), on='user_id_raw', how='inner')
    .join(item_counts.select('item_id_raw'), on='item_id_raw', how='inner')
)
print(f'After sparse filtering: {clean_df.count():,}')
clean_df.show(5, truncate=30)

In [ ]:
from pyspark.sql.window import Window

# ── 4e. String → Integer ID encoding (required by ALS) ────────────────────
user_indexer = StringIndexer(inputCol='user_id_raw', outputCol='user_id', handleInvalid='keep')
item_indexer = StringIndexer(inputCol='item_id_raw', outputCol='item_id', handleInvalid='keep')

pipeline = Pipeline(stages=[user_indexer, item_indexer])
indexer_model = pipeline.fit(clean_df)

indexed_df = indexer_model.transform(clean_df)

# Cast to IntegerType (ALS requires integer IDs, not double)
indexed_df = (
    indexed_df
    .withColumn('user_id', F.col('user_id').cast(IntegerType()))
    .withColumn('item_id', F.col('item_id').cast(IntegerType()))
)

# Save the mapping tables for later use in streaming
user_map = indexer_model.stages[0].labels  # list: index → original_id
item_map = indexer_model.stages[1].labels

# Save reverse maps as parquet for the streaming notebook
user_map_df = spark.createDataFrame(
    [(i, uid) for i, uid in enumerate(user_map)], ['user_id', 'user_id_raw']
)
item_map_df = spark.createDataFrame(
    [(i, iid) for i, iid in enumerate(item_map)], ['item_id', 'item_id_raw']
)
user_map_df.write.mode('overwrite').parquet('/data/user_map')
item_map_df.write.mode('overwrite').parquet('/data/item_map')

print(f'Unique users: {indexed_df.select("user_id").distinct().count():,}')
print(f'Unique items: {indexed_df.select("item_id").distinct().count():,}')
indexed_df.select('user_id', 'item_id', 'rating', 'timestamp').show(5)

## Step 5 — User Segmentation (Personalization Focus)

We segment users into 3 groups based on behavior. This helps us:
1. Apply different recommendation strategies per segment
2. Handle cold-start differently for new vs. active users
3. Provide richer analytics in the dashboard

| Segment | Criteria | Strategy |
|---------|----------|----------|
| `power_user` | ≥ 20 ratings | Full ALS personalization |
| `regular_user` | 10–19 ratings | ALS + popularity boost |
| `cold_start` | 5–9 ratings | Segment-based popularity fallback |

In [ ]:
# Compute per-user activity stats
user_stats = (
    indexed_df
    .groupBy('user_id', 'user_id_raw')
    .agg(
        F.count('rating').alias('num_ratings'),
        F.avg('rating').alias('avg_rating'),
        F.stddev('rating').alias('std_rating'),
    )
    .withColumn(
        'segment',
        F.when(F.col('num_ratings') >= 20, 'power_user')
         .when(F.col('num_ratings') >= 10, 'regular_user')
         .otherwise('cold_start')
    )
)

# Save segments for streaming lookup
user_stats.write.mode('overwrite').parquet('/data/user_segments')

# Show distribution
print('=== User Segment Distribution ===')
user_stats.groupBy('segment').count().orderBy('count', ascending=False).show()

# Join segments back to main df
indexed_df = indexed_df.join(
    user_stats.select('user_id', 'segment'),
    on='user_id', how='left'
)

## Step 6 — Train/Test Split & ALS Training

We use an **80/20 stratified split**. Spark's `randomSplit` is used per segment
to ensure each segment is proportionally represented in both sets.

### ALS Parameters
| Parameter | Value | Meaning |
|-----------|-------|---------|
| `rank` | 20 | Latent factor dimensions |
| `maxIter` | 15 | Training iterations |
| `regParam` | 0.1 | L2 regularization (prevents overfitting) |
| `coldStartStrategy` | `drop` | Ignore unknown users in evaluation |

In [ ]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Persist before split to avoid recomputation
ratings_df = indexed_df.select('user_id', 'item_id', 'rating').cache()

# 80/20 split
train_df, test_df = ratings_df.randomSplit([0.8, 0.2], seed=42)
print(f'Train size: {train_df.count():,}  |  Test size: {test_df.count():,}')

# ── Define ALS model ─────────────────────────────────────────────────────
als = ALS(
    rank=20,
    maxIter=15,
    regParam=0.1,
    userCol='user_id',
    itemCol='item_id',
    ratingCol='rating',
    coldStartStrategy='drop',    # Drop rows where user/item is unknown during eval
    implicitPrefs=False,         # Explicit ratings (not implicit clicks)
    nonnegative=True,            # Keep factors non-negative (improves interpretability)
    seed=42
)

print('Training ALS model...')
als_model = als.fit(train_df)
print('Training complete!')

## Step 7 — RMSE Evaluation & Tuning

We compute RMSE on the test set. If RMSE > 1.5 we must tune.
Typical Amazon review RMSE lands between 0.9 – 1.3 with good parameters.

In [ ]:
evaluator = RegressionEvaluator(
    metricName='rmse',
    labelCol='rating',
    predictionCol='prediction'
)

predictions = als_model.transform(test_df)
rmse = evaluator.evaluate(predictions)
print(f'\n=== Initial ALS RMSE: {rmse:.4f} ===')

if rmse > 1.5:
    print('RMSE > 1.5 — Tuning parameters...')
    for rank in [10, 30, 50]:
        for reg in [0.01, 0.05, 0.15]:
            als_tune = ALS(
                rank=rank, maxIter=15, regParam=reg,
                userCol='user_id', itemCol='item_id', ratingCol='rating',
                coldStartStrategy='drop', nonnegative=True, seed=42
            )
            m = als_tune.fit(train_df)
            r = evaluator.evaluate(m.transform(test_df))
            print(f'  rank={rank}, regParam={reg} → RMSE={r:.4f}')
            if r < rmse:
                rmse = r
                als_model = m
                print(f'  ✓ New best model!')
    print(f'\nBest RMSE after tuning: {rmse:.4f}')
else:
    print('✓ RMSE is within acceptable range — no tuning needed.')

## Step 8 — Save Model & Popularity Fallback

We save:
1. The ALS model (used by the streaming notebook for recommendations)
2. A popularity fallback table (top-50 items by rating count + avg rating)
   — This is the **cold-start fallback** for users with < 5 interactions

In [ ]:
# Save ALS model
MODEL_PATH = '/data/als_model'
als_model.write().overwrite().save(MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')

# Save the indexer pipeline (needed to encode streaming user/item IDs)
indexer_model.write().overwrite().save('/data/indexer_pipeline')
print('Indexer pipeline saved.')

# ── Popularity fallback table for cold-start users ────────────────────────
popularity_df = (
    indexed_df
    .groupBy('item_id', 'item_id_raw')
    .agg(
        F.count('rating').alias('num_ratings'),
        F.avg('rating').alias('avg_rating'),
        F.sum('rating').alias('sum_rating')
    )
    # Bayesian average: (count * avg + C * m) / (count + C)
    # where C=50 (smoothing), m=global mean
    .withColumn('global_mean', F.lit(indexed_df.agg(F.avg('rating')).first()[0]))
    .withColumn(
        'bayesian_score',
        (F.col('num_ratings') * F.col('avg_rating') + F.lit(50) * F.col('global_mean'))
        / (F.col('num_ratings') + F.lit(50))
    )
    .orderBy(F.col('bayesian_score').desc())
    .limit(200)  # Keep top 200 popular items
)

popularity_df.write.mode('overwrite').parquet('/data/popularity_fallback')
print('Popularity fallback table saved.')
popularity_df.show(10, truncate=30)

print('\n=== Notebook 1 Complete ===')
print(f'Final RMSE: {rmse:.4f}')
print('Run Notebook 2 (Kafka Producer) and Notebook 3 (Streaming) next.')

In [ ]:
# ── Summary Statistics for Report ────────────────────────────────────────
import matplotlib.pyplot as plt
import pandas as pd

# Rating distribution
rating_dist = indexed_df.groupBy('rating').count().orderBy('rating').toPandas()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
axes[0].bar(rating_dist['rating'], rating_dist['count'], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Segment distribution
seg_dist = user_stats.groupBy('segment').count().toPandas()
axes[1].pie(seg_dist['count'], labels=seg_dist['segment'],
            autopct='%1.1f%%', colors=['#2196F3','#FF9800','#4CAF50'])
axes[1].set_title('User Segments', fontsize=13, fontweight='bold')

# Ratings per user histogram
user_rating_counts = user_stats.select('num_ratings').toPandas()
axes[2].hist(user_rating_counts['num_ratings'].clip(upper=100), bins=30,
             color='coral', edgecolor='white')
axes[2].set_title('Ratings per User (capped at 100)', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Number of ratings')
axes[2].set_ylabel('Users')

plt.suptitle(f'Amazon Electronics — Dataset Analysis  |  RMSE: {rmse:.4f}',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('/data/dataset_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to /data/dataset_analysis.png')